<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/feature/marquise-transcoder-feature-inspection/notebooks/pi05_e4_time_conditioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# E4 — What the flow-time conditioning buys, and what it costs

**Question.** Each transcoder is conditioned on the flow timestep, and a feature
is identified by the triple (layer, tau, index), so a circuit is traced over ten
times the dictionary. Does that conditioning change what the code represents, or
only how it is parameterised? And how redundant is the node space it creates?

**Why this is cheap.** It reads a feature-discovery run that already exists. No
policy, no simulator, no GPU, no LeRobot install. Colab's own torch and numpy are
enough, so this notebook runs in about a minute.

**Decision rule, fixed in advance.** A layer uses its conditioning when the
correlation between its first and last flow time falls below half of what smooth
drift predicts from its own adjacent-step correlation. Drift alone predicts
`r_adjacent ** (n_steps - 1)`; observing far less than that means the code is
restructured with tau, not merely interpolated. Layers whose mean estimates are
too noisy to support the comparison are reported as unmeasurable rather than
corrected.

**Output.** A per-layer table, a figure, and a verdict, written next to the
discovery run and copied to Drive. Paste the printed verdict block back for the
write-up.


In [ ]:
# @title Controls

# Drive holds the feature-discovery runs.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}

# A feature discovery directory (the one holding feature_stats.pt). Leave blank
# to use the most recent one found under DRIVE_ROOT/outputs/features.
FEATURE_DIR = ""  # @param {type:"string"}

REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/marquise-transcoder-feature-inspection"  # @param {type:"string"}

SAVE_TO_DRIVE = True  # @param {type:"boolean"}
print("Drive root:", DRIVE_ROOT)


In [ ]:
# @title Mount Drive And Clone

import subprocess, time
from pathlib import Path
from google.colab import drive


def mount_with_retry(mountpoint="/content/drive", attempts=3):
    if Path(mountpoint, "MyDrive").exists():
        print("Drive already mounted"); return
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted (attempt {attempt})"); return
        except Exception as exc:
            print(f"attempt {attempt}/{attempts} failed: {exc}")
            if attempt < attempts:
                time.sleep(5 * attempt)
    raise RuntimeError("Could not mount Drive. Runtime > Manage sessions, end other sessions, retry.")


mount_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)

LOCAL_REPO = Path("/content/pi05-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)
subprocess.run(["git", "-C", str(LOCAL_REPO), "log", "-1", "--oneline"], check=True)


In [ ]:
# @title Locate The Discovery Run

from pathlib import Path

if FEATURE_DIR.strip():
    feature_dir = Path(FEATURE_DIR.strip())
else:
    roots = [DRIVE_ROOT / "outputs/features", Path("/content/pi05-run/outputs/features")]
    found = [p.parent for root in roots if root.exists() for p in root.rglob("feature_stats.pt")]
    if not found:
        raise FileNotFoundError(
            "No feature_stats.pt under " + " or ".join(str(r) for r in roots) + ".\n"
            "Run the circuit-trace stage of the counterfactual notebook first, or set FEATURE_DIR."
        )
    feature_dir = max(found, key=lambda p: (p / "feature_stats.pt").stat().st_mtime)
    if len(found) > 1:
        print(f"{len(found)} discovery runs found; using the most recent.")

stats = feature_dir / "feature_stats.pt"
print("discovery run:", feature_dir)
print("feature_stats.pt:", f"{stats.stat().st_size / 1024**2:.0f} MB")


In [ ]:
# @title Run The Analysis

import os, subprocess, sys
from pathlib import Path

out_dir = Path("/content/e4_time_conditioning")
out_dir.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
cmd = [sys.executable, "-u", "scripts/analyze_tau_conditioning.py", str(feature_dir), "--output-dir", str(out_dir)]
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise SystemExit(f"analysis exited {proc.returncode}")


In [ ]:
# @title Table, Figure And Verdict

import json
from IPython.display import display, HTML, Markdown
import matplotlib.pyplot as plt

report = json.loads((out_dir / "tau_conditioning.json").read_text())
rows, summary = report["layers"], report["summary"]

def _f(v, spec=".3f"):
    return "n/a" if v is None else (format(float(v), spec) if isinstance(v, (int, float)) else str(v))

display(Markdown(f"### Flow-time structure by layer ({summary['observations']} observations)"))
display(HTML(
    "<table><tr><th>layer</th><th>live</th><th>reliability</th><th>adjacent r</th>"
    "<th>distant r</th><th>drift predicts</th><th>ratio</th><th>uses tau</th></tr>"
    + "".join(
        f"<tr><td>{r['layer_index']}</td><td>{r.get('features_live','')}</td>"
        f"<td>{_f(r.get('reliability_min'),'.2f')}</td><td>{_f(r.get('adjacent_r'))}</td>"
        f"<td>{_f(r.get('far_r'))}</td><td>{_f(r.get('far_r_predicted_by_drift'))}</td>"
        f"<td>{_f(r.get('structure_ratio'),'.2f')}</td>"
        f"<td>{'yes' if r.get('uses_conditioning') else ('&mdash;' if r.get('usable') else 'not measurable')}</td></tr>"
        for r in rows)
    + "</table>"
))

usable = [r for r in rows if r.get("usable")]
if usable:
    idx = [r["layer_index"] for r in usable]
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(idx, [r["adjacent_r"] for r in usable], "o-", label="adjacent flow times")
    ax.plot(idx, [r["far_r_predicted_by_drift"] for r in usable], "--",
            color="grey", label="smooth-drift prediction for distant")
    ax.plot(idx, [r["far_r"] for r in usable], "s-", label="distant flow times (observed)")
    for r in usable:
        if r["uses_conditioning"]:
            ax.axvspan(r["layer_index"] - 0.5, r["layer_index"] + 0.5, color="tab:green", alpha=0.10)
    ax.set_xlabel("action-expert layer"); ax.set_ylabel("correlation across features")
    ax.set_title("Shaded: conditioning restructures the code beyond drift")
    ax.set_ylim(0, 1.05); ax.legend(loc="lower left", fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(out_dir / "tau_conditioning.png", dpi=150)
    plt.show()

display(Markdown(
    f"### Verdict\n\n**{summary['verdict']}**\n\n"
    f"<small>rule: {summary['decision_rule']}</small>\n\n"
    f"- adjacent-tau r: median {_f(summary['adjacent_r_median'])}, "
    f"range {_f((summary['adjacent_r_range'] or [None,None])[0])} to {_f((summary['adjacent_r_range'] or [None,None])[1])}\n"
    f"- distant-tau r: median {_f(summary['far_r_median'])}, "
    f"range {_f((summary['far_r_range'] or [None,None])[0])} to {_f((summary['far_r_range'] or [None,None])[1])}\n"
    f"- layers using conditioning: {summary['layer_indices_using_conditioning']}\n"
))
print("\nPaste the verdict block above, and tau_conditioning.csv, into the write-up.")


In [ ]:
# @title Save To Drive

import shutil
from pathlib import Path

if SAVE_TO_DRIVE:
    target = DRIVE_ROOT / "outputs/experiments/e4_time_conditioning" / feature_dir.name
    shutil.copytree(out_dir, target, dirs_exist_ok=True)
    print("saved ->", target)
    for p in sorted(target.iterdir()):
        print(f"   {p.name}  ({p.stat().st_size/1024:.0f} KB)")
else:
    print("SAVE_TO_DRIVE is off; results stay at", out_dir, "and are lost when the runtime ends.")
